# Limpieza inicial de noticias descubiertas

Este notebook crea una copia limpia de los metadatos descubiertos por Google News RSS. La base original no se modifica. En esta versión no se filtran noticias por relevancia vial ni se analiza sentimiento.

In [1]:
from pathlib import Path
import re
import unicodedata

from bs4 import BeautifulSoup
import pandas as pd

def encontrar_base_proyecto(inicio):
    for ruta in (inicio, *inicio.parents):
        if (ruta / 'data').is_dir() and (ruta / 'src').is_dir():
            return ruta
    raise FileNotFoundError('No se encontró la raíz del proyecto.')

BASE_DIR = encontrar_base_proyecto(Path.cwd().resolve())
INPUT_FILE = BASE_DIR / 'data' / 'noticias' / 'scraping_noticias' / 'noticias_descubiertas.csv'
OUTPUT_FILE = BASE_DIR / 'data' / 'noticias' / 'scraping_noticias' / 'noticias_limpias.csv'

if not INPUT_FILE.exists():
    raise FileNotFoundError(f'No se encontró la base de noticias: {INPUT_FILE}')


In [2]:
df = pd.read_csv(INPUT_FILE)
print(f'Registros cargados: {len(df)}')
display(df.head())
display(df.isna().sum().rename('valores_nulos').to_frame())

Registros cargados: 435


,id_registro,titulo,url,fecha_publicacion,resumen,consulta,fuente_descubrimiento,fecha_scraping
0,news_00001,Sellan huecos en puentes peatonales que eran u...,https://news.google.com/rss/articles/CBMisAFBV...,"Fri, 18 Sep 2026 17:10:11 GMT","<a href=""https://news.google.com/rss/articles/...","""huecos"" Bogotá",Google News RSS,2026-09-21T11:45:27-05:00
1,news_00002,Ya se han tapado más de 200 mil huecos en los ...,https://news.google.com/rss/articles/CBMinwFBV...,"Wed, 18 Mar 2026 07:00:00 GMT","<a href=""https://news.google.com/rss/articles/...","""huecos"" Bogotá",Google News RSS,2026-09-21T11:45:27-05:00
2,news_00003,Más de 1.400 personas han muerto por siniestro...,https://news.google.com/rss/articles/CBMizAFBV...,"Thu, 23 Jul 2026 07:00:00 GMT","<a href=""https://news.google.com/rss/articles/...","""huecos"" Bogotá",Google News RSS,2026-09-21T11:45:27-05:00
3,news_00004,Huecos en Bogotá dejan 1.409 muertes durante l...,https://news.google.com/rss/articles/CBMi-wFBV...,"Thu, 23 Jul 2026 07:00:00 GMT","<a href=""https://news.google.com/rss/articles/...","""huecos"" Bogotá",Google News RSS,2026-09-21T11:45:27-05:00
4,news_00005,RecWays: la app que “escanea” en tiempo real l...,https://news.google.com/rss/articles/CBMioAFBV...,"Sun, 12 Apr 2026 07:00:00 GMT","<a href=""https://news.google.com/rss/articles/...","""huecos"" Bogotá",Google News RSS,2026-09-21T11:45:27-05:00


,valores_nulos
id_registro,0
titulo,0
url,0
fecha_publicacion,0
resumen,0
consulta,0
fuente_descubrimiento,0
fecha_scraping,0


In [3]:
def limpiar_texto(valor):
    """Quita HTML y normaliza espacios sin cambiar el archivo original."""
    if pd.isna(valor):
        return pd.NA
    texto = BeautifulSoup(str(valor), 'html.parser').get_text(' ')
    return re.sub(r'\s+', ' ', texto).strip()

df_limpio = df.copy()
df_limpio['titulo_limpio'] = df_limpio['titulo'].map(limpiar_texto)
df_limpio['resumen_limpio'] = df_limpio['resumen'].map(limpiar_texto)

df_limpio['fecha_publicacion_estandarizada'] = pd.to_datetime(
    df_limpio['fecha_publicacion'], errors='coerce', utc=True
)
df_limpio['fecha_scraping_estandarizada'] = pd.to_datetime(
    df_limpio['fecha_scraping'], errors='coerce', utc=True
)

registros_antes = len(df_limpio)
df_limpio = df_limpio.drop_duplicates(subset='url', keep='first').reset_index(drop=True)
print(f'Duplicados eliminados por URL: {registros_antes - len(df_limpio)}')
print(f'Registros limpios: {len(df_limpio)}')

Duplicados eliminados por URL: 0
Registros limpios: 435


In [4]:
# Filtro interpretable basado únicamente en título y resumen RSS.
# No reemplaza la revisión posterior con el contenido original de la noticia.
CLAVES_DETERIORO = ['hueco', 'bache', 'malla vial', 'deterioro vial', 'asfalto', 'pavimento']
CONTEXTO_VIAL = ['calle', 'carrera', 'avenida', 'via', 'vial', 'movilidad', 'umv', 'idu', 'bogota']
EXCLUSIONES = ['hueco fiscal']

def normalizar_para_busqueda(texto):
    texto = '' if pd.isna(texto) else str(texto)
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('ascii')
    return texto.casefold()

def evaluar_relevancia_preliminar(fila):
    texto = normalizar_para_busqueda(f"{fila['titulo_limpio']} {fila['resumen_limpio']}")
    deterioro = [clave for clave in CLAVES_DETERIORO if clave in texto]
    contexto = [clave for clave in CONTEXTO_VIAL if clave in texto]
    exclusiones = [clave for clave in EXCLUSIONES if clave in texto]
    score = len(deterioro) + len(contexto)
    candidata = bool(deterioro and contexto and not exclusiones)
    motivos = (
        f"deterioro: {', '.join(deterioro) or 'ninguno'} | "
        f"contexto: {', '.join(contexto) or 'ninguno'} | "
        f"exclusiones: {', '.join(exclusiones) or 'ninguna'}"
    )
    return pd.Series([score, candidata, motivos])

df_limpio[['score_relevancia_preliminar', 'es_candidata_vial_preliminar', 'motivos_relevancia_preliminar']] = (
    df_limpio.apply(evaluar_relevancia_preliminar, axis=1)
)
print(df_limpio['es_candidata_vial_preliminar'].value_counts(dropna=False))

es_candidata_vial_preliminar
False    220
True     215
Name: count, dtype: int64


In [5]:
columnas_revision = [
    'titulo_limpio', 'resumen_limpio', 'score_relevancia_preliminar',
    'es_candidata_vial_preliminar', 'motivos_relevancia_preliminar'
]
display(df_limpio[columnas_revision].head())
display(df_limpio[columnas_revision].isna().sum().rename('valores_nulos').to_frame())

,titulo_limpio,resumen_limpio,score_relevancia_preliminar,es_candidata_vial_preliminar,motivos_relevancia_preliminar
0,Sellan huecos en puentes peatonales que eran u...,Sellan huecos en puentes peatonales que eran u...,2,True,deterioro: hueco | contexto: bogota | exclusio...
1,Ya se han tapado más de 200 mil huecos en los ...,Ya se han tapado más de 200 mil huecos en los ...,2,True,deterioro: hueco | contexto: bogota | exclusio...
2,Más de 1.400 personas han muerto por siniestro...,Más de 1.400 personas han muerto por siniestro...,3,True,"deterioro: hueco | contexto: via, bogota | exc..."
3,Huecos en Bogotá dejan 1.409 muertes durante l...,Huecos en Bogotá dejan 1.409 muertes durante l...,4,True,"deterioro: hueco | contexto: via, vial, bogota..."
4,RecWays: la app que “escanea” en tiempo real l...,RecWays: la app que “escanea” en tiempo real l...,2,True,deterioro: hueco | contexto: bogota | exclusio...


,valores_nulos
titulo_limpio,0
resumen_limpio,0
score_relevancia_preliminar,0
es_candidata_vial_preliminar,0
motivos_relevancia_preliminar,0


In [6]:
df_limpio.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f'Archivo limpio guardado en: {OUTPUT_FILE}')

Archivo limpio guardado en: C:\Users\Usuario\Documents\proyecto_malla_vial_bogota\data\noticias\scraping_noticias\noticias_limpias.csv
